In [2]:
#调用模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

### 工具信息
##### 当模型进行工具调用时，它们包含在 AIMessage 中

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-nano")

def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

print(response)
print(response.tool_calls)

Tool: get_weather
Args: {'location': 'Paris'}
ID: call_AyACFe1TFh1qlNfcAREzIrKY
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 152, 'prompt_tokens': 131, 'total_tokens': 283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXMD1X8yFFCNoYtjW9sHepOVxpNIA', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--2262ba4c-b234-4567-b05c-17e55acad091-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Paris'}, 'id': 'call_AyACFe1TFh1qlNfcAREzIrKY', 'type': 'tool_call'}] usage_metadata={'input_tokens': 131, 'output_tokens': 152, 'total_tokens': 283, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}

### 工具信息
##### 对于支持工具调用的模型，AI 消息可以包含工具调用。工具消息用于将单个工具执行的结果传递回模型。

In [8]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

# After a model makes a tool call
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result(此消息响应的工具调用的 ID。（这必须与 AIMessage 中工具调用的 ID 匹配))
]
response = model.invoke(messages)  # Model processes the result
print(response)

content="Right now in San Francisco it's sunny and about 72°F (roughly 22°C). Want an hourly forecast or conditions for the next few days?" additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 233, 'prompt_tokens': 47, 'total_tokens': 280, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-CXMVaysXKr37PXXEVtBPLvIcYGqeH', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--7ca432f0-ec3e-43aa-ac97-bcc018e5da29-0' usage_metadata={'input_tokens': 47, 'output_tokens': 233, 'total_tokens': 280, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 192}}


## 消息类型

### 多模态输入

In [ ]:
from langchain.messages import HumanMessage

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

### 标准内容表达方式
##### LangChain 提供了一种标准的消息内容表示方式，可跨不同提供商使用。Message 对象实现了一个属性，它会惰性（lazily）地将该属性解析成一种标准、类型安全的表示形式。例如，由 ChatAnthropic 或 ChatOpenAI 生成的消息会分别包含各自提供商格式的 thinking 或 reasoning 块，但这些块可以被惰性解析为统一的 ReasoningContentBlock 表示。
##### 大致就是集成

In [9]:
#Anthropic
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {"type": "thinking", "thinking": "...", "signature": "WaUjzkyp..."},
        {"type": "text", "text": "..."},
    ],
    response_metadata={"model_provider": "anthropic"}
)

print(message.content_blocks)

[{'type': 'reasoning', 'reasoning': '...', 'extras': {'signature': 'WaUjzkyp...'}}, {'type': 'text', 'text': '...'}]


In [10]:
#OpenAI
from langchain.messages import AIMessage

message = AIMessage(
    content=[
        {
            "type": "reasoning",
            "id": "rs_abc123",
            "summary": [
                {"type": "summary_text", "text": "summary 1"},
                {"type": "summary_text", "text": "summary 2"},
            ],
        },
        {"type": "text", "text": "...", "id": "msg_abc123"},
    ],
    response_metadata={"model_provider": "openai"}
)

print(message.content_blocks)

[{'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 1'}, {'type': 'reasoning', 'id': 'rs_abc123', 'reasoning': 'summary 2'}, {'type': 'text', 'text': '...', 'id': 'msg_abc123'}]


### 多模态

##### 图像输入

In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "url": "https://example.com/path/to/image.jpg"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {
            "type": "image",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",   #将图像转换为base64编码（文本）传入
            "mime_type": "image/jpeg",                                   #声明base64编码的mime类型（传入的原始文件的类型）
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this image."},
        {"type": "image", "file_id": "file-abc123"},                    #根据先前传入的文件的file_id获取图片
    ]
}

##### PDF文档输入

In [ ]:
# From URL
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "url": "https://example.com/path/to/document.pdf"},
    ]
}

# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {
            "type": "file",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "application/pdf",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this document."},
        {"type": "file", "file_id": "file-abc123"},
    ]
}

##### 音频输入

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {
            "type": "audio",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "audio/wav",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this audio."},
        {"type": "audio", "file_id": "file-abc123"},
    ]
}

##### 视频输入

In [ ]:
# From base64 data
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {
            "type": "video",
            "base64": "AAAAIGZ0eXBtcDQyAAAAAGlzb21tcDQyAAACAGlzb2...",
            "mime_type": "video/mp4",
        },
    ]
}

# From provider-managed File ID
message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "Describe the content of this video."},
        {"type": "video", "file_id": "file-abc123"},
    ]
}